# Clase 24 — Bonus track: cómo funciona RAG por dentro

**Diplomado IA Aplicada al Diseño · UDD · 2026** · Prof. Darío Osorio

Este es el **último notebook de Colab del curso**. Es opcional pero recomendado. Muestra en 20 minutos cómo funciona por dentro el patrón que usa NotebookLM (y muchas otras herramientas).

Después de esta clase pasamos 100% a herramientas SaaS.

## Objetivo
Construir un mini-RAG con 3 documentos ficticios y hacerle preguntas.

In [ ]:
!pip install -q sentence-transformers numpy
print("Listo.")

Listo.


In [ ]:
!pip install -q PyPDF2
print("Listo.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 10.1 MB/s eta 0:00:00
Listo.


## Paso 1 — Nuestros "documentos"

En NotebookLM cargas PDFs. Acá simulamos con strings.

In [ ]:
import PyPDF2

# 1. Define las rutas de los archivos que quieres leer.
# TIP: Haz clic en los 3 puntitos al lado del archivo en el panel izquierdo y selecciona "Copiar ruta" (Copy path).
rutas_pdfs = [
    "App para Clínica Veterinaria Eficiente.pdf",  # Reemplaza por el nombre exacto y extensión de tu archivo
    "2026 Global Human Capital Trends DELOITTE.pdf" # Reemplaza por el nombre exacto y extensión de tu archivo
    "VetFlow AI _ Google AI Studio.html"
    "EJERCICIO-02B_BBCG.docx"
    "EJERCICIO-02A_Barrera-Blau-Gouet-Cid.docx"
]

documentos = []

# 2. Recorremos cada archivo PDF para extraer su texto
for ruta in rutas_pdfs:
    texto_acumulado = ""
    try:
        # Abrimos el archivo en modo de lectura binaria ('rb')
        with open(ruta, 'rb') as archivo_pdf:
            lector = PyPDF2.PdfReader(archivo_pdf)

            # Extraemos el texto página por página
            for pagina in lector.pages:
                texto_pagina = pagina.extract_text()
                if texto_pagina:
                    texto_acumulado += texto_pagina + "\n"

        # Añadimos el texto completo del PDF a nuestra lista de documentos
        documentos.append(texto_acumulado)
        print(f"✔ Cargado exitosamente: {ruta}")

    except FileNotFoundError:
        print(f"❌ Error: No se encontró el archivo en la ruta {ruta}")
    except Exception as e:
        print(f"❌ Error al leer {ruta}: {e}")

print("\n---")
print(f"{len(documentos)} documentos PDF cargados en total y listos para convertirse a embeddings.")

✔ Cargado exitosamente: App para Clínica Veterinaria Eficiente.pdf
❌ Error: No se encontró el archivo en la ruta 2026 Global Human Capital Trends DELOITTE.pdfVetFlow AI _ Google AI Studio.htmlEJERCICIO-02B_BBCG.docxEJERCICIO-02A_Barrera-Blau-Gouet-Cid.docx

---
1 documentos PDF cargados en total y listos para convertirse a embeddings.


## Paso 2 — Convertir documentos a embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

modelo = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = modelo.encode(documentos)
print(f"Shape: {embeddings.shape}")
print("Cada documento es ahora un vector de 384 dimensiones.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape: (1, 384)
Cada documento es ahora un vector de 384 dimensiones.


## Paso 3 — Función de búsqueda semántica

Le damos una pregunta, ella busca el documento más parecido.

In [ ]:
def buscar(pregunta, top_k=1):
    p_emb = modelo.encode([pregunta])
    sims = np.dot(embeddings, p_emb.T).flatten()
    sims = sims / (np.linalg.norm(embeddings, axis=1) * np.linalg.norm(p_emb))
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(documentos[i], float(sims[i])) for i in top_idx]

preguntas = [
    "que problemas de seguridad mencionan?",
    "hay algo sobre los iconos?",
    "que dijeron los adultos mayores?",
]

for q in preguntas:
    print(f"P: {q}")
    for doc, sim in buscar(q, top_k=1):
        print(f"  Similitud: {sim:.2f}")
        print(f"  Documento: {doc[:150]}...")
    print()

P: que problemas de seguridad mencionan?
  Similitud: 0.34
  Documento: App para Clínica V eterinaria Eﬁciente
https://gemini.google.com/app/f5d1541ce13a0e0d
User prompt: Según el rol de un recepcionista y coordinador de a...

P: hay algo sobre los iconos?
  Similitud: 0.20
  Documento: App para Clínica V eterinaria Eﬁciente
https://gemini.google.com/app/f5d1541ce13a0e0d
User prompt: Según el rol de un recepcionista y coordinador de a...

P: que dijeron los adultos mayores?
  Similitud: 0.28
  Documento: App para Clínica V eterinaria Eﬁciente
https://gemini.google.com/app/f5d1541ce13a0e0d
User prompt: Según el rol de un recepcionista y coordinador de a...



## Cierre

Esto es exactamente lo que hace NotebookLM (más pulido y con Gemini razonando encima). Ahora que viste el mecanismo, ya sabes por qué:

- Cuando la respuesta "no está en las fuentes", el sistema puede decir "no sé".
- Las citas son literales (vienen del documento recuperado).
- La calidad depende del corte (chunking) y de qué tan buenas son tus fuentes.

**No hace falta que ejecutes esto en tu proyecto** — usa NotebookLM directamente. Este notebook es solo para entender qué pasa por debajo.